# Object Recognition and Classification in Noisy Environments

This project presents a complete image processing pipeline for extracting, indexing, and classifying industrial objects from a real-world image with challenging lighting conditions (shadows, glare).

The algorithm automatically separates screws, nuts, and washers based on their defined physical and mathematical features.

### Solution Architecture
1. **Background estimation and compensation:** Large-kernel morphological closing models the averaged illumination background, followed by hardware-accelerated subtraction to normalize object luminance.
2. **Segmentation and denoising:** Global thresholding combined with iterative morphological filters (opening + closing) removes internal reflections (holes) and separates objects from the background.
3. **Invariant geometric feature extraction (feature engineering):**
   - Computation of *Moments* and *Hu Moments* to capture elongation, typical of screws.
   - Area analysis from Connected Components Statistics to distinguish large washers from smaller nuts.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob

# Download raw industrial image
if not os.path.exists("details.png"):
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/13_CCL/details.png --no-check-certificate

im = cv2.imread('details.png', cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(10,6))
plt.imshow(im, 'gray')
plt.title("Input Image (Uneven Illumination and Shadows Visible)")
plt.axis('off')
plt.show()

## Classification Pipeline Implementation

In [ ]:
# 1. Background gradient modeling and removal (background subtraction)
bg = cv2.morphologyEx(im, cv2.MORPH_CLOSE, np.ones((51, 51), np.uint8))
diff = cv2.subtract(bg, im)

# 2. Binary segmentation
_, th = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

# 3. Topology cleanup (morphology)
th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))  # Remove speckle noise
th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8)) # Fill holes in structures

# 4. Indexing and statistics extraction (connected component labeling)
n, ccl, stats, _ = cv2.connectedComponentsWithStats(th)

# Initialize classifier output buffers
sr = np.zeros_like(im)
na = np.zeros_like(im)
po = np.zeros_like(im)

# 5. Object classifier loop
for i in range(1, n):
    area = stats[i, cv2.CC_STAT_AREA]

    # Ignore false-positive detections
    if area < 100:
        continue

    # Pixel mask of the current object
    msk = (ccl == i).astype(np.uint8)
    m = cv2.moments(msk)
    
    # Guard against division by zero
    if m['m00'] == 0: 
        continue

    # Compute 7-dimensional Hu moment vector
    hu = cv2.HuMoments(m).flatten()
    
    # Decision logic (hard-coded classifier)
    # Feature 1: first Hu moment > 0.2 identifies highly elongated structures (screws)
    if hu[0] > 0.2:
        sr[msk == 1] = 255
    # Feature 2: area-based separation for axially symmetric parts (washers vs nuts)
    elif area > 1500:
        po[msk == 1] = 255
    else:
        na[msk == 1] = 255

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(sr, cmap="gray")
ax[0].axis("off")
ax[0].set_title("Class: Screws (High Hu[0] Elongation)")

ax[1].imshow(na, cmap="gray")
ax[1].axis("off")
ax[1].set_title("Class: Nuts (Symmetric, Low Area)")

ax[2].imshow(po, cmap="gray")
ax[2].axis("off")
ax[2].set_title("Class: Washers (Symmetric, High Area)")

plt.tight_layout()
plt.show()

### Workspace Cleanup

In [ ]:
trash_files = glob.glob('details.png')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Temporary resources removed from the local filesystem.")